### 文档切分器 Text Splitters

###  def split_text(self, text: str) -> list[str]:
 用法：传入单个字符串，切分成多个字符串块
 调用关系：抽象方法，由子类实现具体切分逻辑

###  def create_documents(self, texts: list[str], metadatas: list[dict[Any, Any]] | None = None
 用法：传入字符串列表，将每个字符串切分后封装成 Document
 调用关系：底层遍历 texts ，并对每个 text 调用 split_text(text)

### def split_documents(self, documents: Iterable[Document]) -> list[Document]:
用法：传入 Document 集合，取出 page_content 和 metadata 后重新切分成 Document
 调用关系：底层调用 create_documents(texts, metadatas=metadatas) ，再间接调用split_text(text)


### **举例 1：** 字符串文本的分割

In [1]:
# 1. 导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2. 示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者
能够更容易地构建复杂的应用程序。
       """
# 3. 定义字符分割器
splitter = CharacterTextSplitter(
      chunk_size=50,  # 每块大小
      chunk_overlap=5,  # 块与块之间的重复字符数
      # length_function=len,
      separator="",  # 设置为空字符串时，表示禁用分隔符优先
)

# 4. 分割文本
texts = splitter.split_text(text)

# 5. 打印结果
for i, chunk in enumerate(texts):
      print(f" 块 {i + 1}: 长度： {len(chunk)}")
      print(chunk)
      print("-" * 50)

 块 1: 长度： 49
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发
--------------------------------------------------
 块 2: 长度： 23
象，使开发者
能够更容易地构建复杂的应用程序。
--------------------------------------------------


### **举例 2：** 指定分割符

In [6]:
# 1. 导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2. 定义要分割的文本
text = " 这是一个示例文本啊。我们将使用 CharacterTextSplitter 将其分割成小块。分割基于字符数。 "

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3. 定义分割器实例
text_splitter = CharacterTextSplitter(
      chunk_size=30,  # 每个块的最大字符数
      chunk_overlap=5,  # 块之间的重叠字符数
      separator="。",  # 按句号分割优先
)

# 4. 开始分割
chunks = text_splitter.split_text(text)

# 5. 打印效果
for i, chunk in enumerate(chunks):
      print(f" 块 {i + 1}: 长度： {len(chunk)}")
      print(chunk)
      print("-" * 50)

Created a chunk of size 35, which is longer than the specified 30


 块 1: 长度： 9
这是一个示例文本啊
--------------------------------------------------
 块 2: 长度： 35
我们将使用 CharacterTextSplitter 将其分割成小块
--------------------------------------------------
 块 3: 长度： 8
分割基于字符数。
--------------------------------------------------


### **举例 3：** 指定分割符

In [8]:
# 1. 导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2. 定义要分割的文本
text = " 这是第一段文本。这是第二段内容。最后一段结束。 "

# 3. 定义字符分割器
text_splitter = CharacterTextSplitter(
      chunk_size=20,  # 每个块的最大字符数
      chunk_overlap=5,  # 块之间的重叠字符数
      separator="。",  # 按句号分割优先
      keep_separator=True,  # chunk 中是否保留切割符
)

# 4. 分割文本
chunks = text_splitter.split_text(text)

# 5. 打印结果
for i, chunk in enumerate(chunks):
      print(f" 块 {i + 1}: 长度： {len(chunk)}")
      print(chunk)
      print("-" * 50)

 块 1: 长度： 15
这是第一段文本。这是第二段内容
--------------------------------------------------
 块 2: 长度： 8
。最后一段结束。
--------------------------------------------------


## RecursiveCharacterTextSplitter ( 递归字符文本切分器 )
尽量保留语义完整性，同时把片段控制在 embedding 与检索更友好的长度范围
内。

### **举例 1：** 使用 split_text() 方法演示

In [9]:
# 1. 导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 定义 RecursiveCharacterTextSplitter 分割器对象
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=10,
      chunk_overlap=0,
      add_start_index=True,
)

# 3. 定义拆分的内容
text = "LangChain 框架特性 \n\n 多模型集成 (GPT/Claude)\n 记忆管理功能 \n 链式调用设计。文档分析场景示例：需要处理 PDF/Word 等格式。 "

# 4. 拆分器分割
paragraphs = text_splitter.split_text(text)

for i, chunk in enumerate(paragraphs):
      print(f" 块 {i + 1}, 长度： {len(chunk)}")
      print(chunk)
      print('-' * 50)

 块 1, 长度： 9
LangChain
--------------------------------------------------
 块 2, 长度： 4
框架特性
--------------------------------------------------
 块 3, 长度： 5
多模型集成
--------------------------------------------------
 块 4, 长度： 9
(GPT/Clau
--------------------------------------------------
 块 5, 长度： 3
de)
--------------------------------------------------
 块 6, 长度： 6
记忆管理功能
--------------------------------------------------
 块 7, 长度： 9
链式调用设计。文档
--------------------------------------------------
 块 8, 长度： 10
分析场景示例：需要处
--------------------------------------------------
 块 9, 长度： 1
理
--------------------------------------------------
 块 10, 长度： 8
PDF/Word
--------------------------------------------------
 块 11, 长度： 4
等格式。
--------------------------------------------------


 **举例 2：** 使用 create_documents() 方法演示，传入字符串列表，返回 Document 对象列表

In [13]:
# 1. 导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 定义 RecursiveCharacterTextSplitter 分割器对象
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=10,
      chunk_overlap=0,
      add_start_index=True,
)

# 3. 定义分割的内容
# text="LangChain 框架特性 \n\n 多模型集成 (GPT/Claude)\n 记忆管理功能 \n 链式调用设计。文档分析场景示例：需要处理 PDF/Word 等格式。 "

list = [
      "LangChain 框架特性 \n\n 多模型集成 (GPT/Claude)\n 记忆管理功能 \n 链式调用设计。文档分析场景示例：需要处理 PDF/Word 等格式。 "
]

# 4. 分割器分割
# create_documents() ：形参是字符串列表，返回值是 Document 的列表
paragraphs = text_splitter.create_documents(list)

for para in paragraphs:
      print(para)
      print('-------')

page_content='LangChain' metadata={'start_index': 0}
-------
page_content='框架特性' metadata={'start_index': 10}
-------
page_content='多模型集成' metadata={'start_index': 18}
-------
page_content='(GPT/Clau' metadata={'start_index': 24}
-------
page_content='de)' metadata={'start_index': 33}
-------
page_content='记忆管理功能' metadata={'start_index': 38}
-------
page_content='链式调用设计。文档' metadata={'start_index': 47}
-------
page_content='分析场景示例：需要处' metadata={'start_index': 56}
-------
page_content='理' metadata={'start_index': 66}
-------
page_content='PDF/Word' metadata={'start_index': 68}
-------
page_content='等格式。' metadata={'start_index': 77}
-------


**举例 3：** 使用 create_documents() 方法演示，将本地文件内容加载成字符串，进行拆分

In [11]:
# 1. 导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 打开 .txt 文件
with open("../asset/load/09-ai.txt", encoding="utf-8") as f:
      state_of_the_union = f.read()  # 返回的是字符串

      # 3. 定义 RecursiveCharacterTextSplitter （递归字符分割器）
      text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=20,
            # chunk_overlap=0,
            length_function=len,
      )

      # 4. 分割文本
      texts = text_splitter.create_documents([state_of_the_union])

      # 5. 打印分割文本
      for text in texts:
            print(f" 🔥 {text.page_content}")

 🔥 人工智能（AI）是什么？
 🔥 人工智能（Artificial
 🔥 Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
 🔥 让机器具备感知环境、处理信息并自主行动的能力。
 🔥 1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
 🔥 深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
 🔥 2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
 🔥 金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
 🔥 伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。
 🔥 未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。


**举例 4：** 使用 split_documents() 方法演示，利用 PDFLoader 加载文档，对文档的内容用递归切割器切割

In [ ]:
# 1. 导入相关依赖
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 定义 PyPDFLoader 加载器
loader = PyPDFLoader("../asset/load/04-load.pdf")

# 3. 加载和切割文档对象
docs = loader.load()  # 返回 Document 对象构成的 list
# print(f" 第 0 页： \n{docs[0]}")

# 4. 定义切割器
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=200,
      # chunk_size=120,
      chunk_overlap=0,
      # chunk_overlap=100,
      length_function=len,
      add_start_index=True,
)
# 5. 对 pdf 内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)

for para in paragraphs:
      print(para)
      print('-------')

**举例 5：** 自定义分隔符

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=200,
      chunk_overlap=20,  # 增加重叠字符
      separators=["\n\n", "\n", " 。 ", " ！ ", " ？ ", "……", " ， ", ""],  # 添加中文标点
      length_function=len,
      keep_separator=True  # 保留句尾标点（如 …… ），避免切割后丢失语气和逻辑
)

### TokenTextSplitter
Token 数量 + 自然边界     典型场景 ：需要精确控制 Token 数输入 LLM 的场景

In [14]:
# 1. 导入相关依赖
from langchain_text_splitters import TokenTextSplitter

# 2. 初始化 TokenTextSplitter
text_splitter = TokenTextSplitter(
      chunk_size=33,  # 最大 token 数为 33
      chunk_overlap=0,  # 重叠 token 数为 0
      # model_name="gpt-4", # 选择 GPT-4 模型的编码器
      encoding_name="cl100k_base",  # 使用 OpenAI 的编码器 , 将文本转换为 token 序列
)
# 3. 定义文本
text = " 人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自 20 世纪 50 年代诞生以来，人工智能经历了多次起伏。 "

# 4. 开始切割
texts = text_splitter.split_text(text)

# 打印分割结果
print(f" 原始文本被分割成了 {len(texts)} 个块 :")
for i, chunk in enumerate(texts):
      print(f" 块 {i + 1}: 长度： {len(chunk)} 内容： {chunk}")
      print("-" * 50)

 原始文本被分割成了 3 个块 :
 块 1: 长度： 29 内容：  人工智能是一个强大的开发框架。它支持多种语言模型和工具链
--------------------------------------------------
 块 2: 长度： 32 内容： 。人工智能是指通过计算机程序模拟人类智能的一门科学。自 20 世
--------------------------------------------------
 块 3: 长度： 25 内容： 纪 50 年代诞生以来，人工智能经历了多次起伏。 
--------------------------------------------------


###  SemanticChunker ：语义分块

In [18]:
# pip install langchain_experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 加载文本
with open("../asset/load/09-ai1.txt", encoding="utf-8") as f:
      state_of_the_union = f.read()  # 返回字符串

      # 获取嵌入模型
      embedding_model = OllamaEmbeddings(
            model="qwen3-embedding:4b",
            base_url=os.getenv("OLAMA_BASE_URL"),
      )
      # embedding_model = OpenAIEmbeddings(
# model="BAAI/bge-m3", # 付费模型 ID ： Pro/BAAI/bge-m3
# base_url=os.getenv("SILICONFLOW_BASE_URL"),
# api_key=os.getenv("SILICONFLOW_API_KEY"),
# dimensions=1024
# )

# 获取切割器
"""
| 类型 | 原理说明 | 适用场景 |
| :--- | :--- | :--- |
| **percentile** | 计算相邻句子嵌入向量的余弦距离，取**距离分布的第N百分位值**作为阈值，高于此值则分割 | 常规文本（如文章、报告） |

| **standard_deviation** | 以**均值 + N倍标准差**为阈值，识别语义突变点 | 语义变化剧烈的文档（如技术手册） |

| **interquartile** | 用**四分位距（IQR）** 定义异常值边界，超过则分割 | 长文档（如书籍） |

| **gradient** | 基于**嵌入向量变化的梯度**检测分割点（需自定义实现） | 实验性需求 |

"""
text_splitter = SemanticChunker(
      embeddings=embedding_model,
      breakpoint_threshold_type="percentile",  # 断点阈值类型：字面值 [" 百分位数 ", " 标准差 ", " 四分位距 ", " 梯度 "] 选其一
      breakpoint_threshold_amount=65.0,  # 断点阈值数量 ( 极低阈值 → 高分割敏感度 )
      sentence_split_regex=r"(?<=[ 。？！ ])\s+"  # 句子切分正则 : 遇到中文的句号、感叹号、问号（。？！）且后面带有空格时，先将其切分为独立的 “ 句子 ” 。
)

# 切分文档
docs = text_splitter.create_documents(texts=[state_of_the_union])

print(len(docs))
for doc in docs:
      print(f" 🔍 文档 : {doc}")

7
 🔍 文档 : page_content='人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence，AI）作为计算机科学的一个重要分支，近年来取得了突飞猛进的发展。本文综述了人工智能的发展历程、核心技术、应用领域以及未来发展趋势。通过对人工智能的定义、历史背景、主要技术（如机器学习、深度学习、自然语言处理等）的详细介绍，探讨了人工智能在医疗、金融、教育、交通等领域的应用，并分析了人工智能发展过程中面临的挑战与机遇。最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方向。 1. 引言
人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏，近年来随着计算能力的提升和大数据的普及，人工智能技术取得了显著的进展。人工智能的应用已经渗透到日常生活的方方面面，从智能手机的语音助手到自动驾驶汽车，从医疗诊断到金融分析，人工智能正在改变着人类社会的运行方式。'
 🔍 文档 : page_content='2. 人工智能的发展历程
2.1 早期发展
人工智能的概念最早可以追溯到20世纪50年代。1956年，达特茅斯会议（Dartmouth Conference）被认为是人工智能研究的正式开端。在随后的几十年里，人工智能研究经历了多次高潮与低谷。早期的研究主要集中在符号逻辑和专家系统上，但由于计算能力的限制和算法的不足，进展缓慢。 2.2 机器学习的兴起
20世纪90年代，随着统计学习方法的引入，机器学习逐渐成为人工智能研究的主流。支持向量机（SVM）、决策树、随机森林等算法在分类和回归任务中取得了良好的效果。这一时期，机器学习开始应用于数据挖掘、模式识别等领域。 2.3 深度学习的突破
2012年，深度学习在图像识别领域取得了突破性进展，标志着人工智能进入了一个新的阶段。深度学习通过多层神经网络模拟人脑的工作方式，能够自动提取特征并进行复杂的模式识别。卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）等深度学习模型在图像处理、自然语言处理、语音识别等领域取得了显著成果。 3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记